In [1]:
# Установка зависимостей (запустите один раз)
# %pip install -r requirements.txt

### Простой агент

Этот ноутбук запускает ReAct-агента
Бэкенд (GigaChat или LM Studio) настраивается в `config.yaml`.

In [2]:
# Импорты
import yaml
from pprint import pprint
from langgraph.prebuilt import create_react_agent
from tools.tools import tools
from connections.clients import get_llm_client

In [3]:
# Загрузка конфигурации
with open('config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print(f"Активный бэкенд: {config['active_backend']}")
pprint(config)

Активный бэкенд: lmstudio
{'active_backend': 'lmstudio',
 'backends': {'gigachat': {'env_vars': {'access_token': 'JPY_API_TOKEN',
                                        'base_url': 'GIGACHAT_API_URL'},
                           'model': 'GigaChat-2'},
              'lmstudio': {'base_url': 'http://localhost:1234/v1',
                           'model': 'openai/gpt-oss-20b'}}}


In [4]:
# Создание LLM клиента
backend = config['active_backend']
llm = get_llm_client(backend, config)
print(f"LLM клиент создан: {type(llm).__name__}")

LLM клиент создан: ChatOpenAI


In [5]:
# Создание ReAct-агента
agent_executor = create_react_agent(
    model=llm,
    tools=tools
)
print("Агент создан успешно!")

Агент создан успешно!


C:\Users\accordij\AppData\Local\Temp\ipykernel_14164\1833556789.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


### Тестовые запросы

In [7]:
# Тест 1: Простое вычисление
query1 = "Сколько будет 52 умножить на 48?"
print(f"Запрос: {query1}\n")

messages = agent_executor.invoke({'messages': [query1]})['messages']
pprint(messages)
print(f"\nОтвет агента: {messages[-1].content}")

Запрос: Сколько будет 52 умножить на 48?

[HumanMessage(content='Сколько будет 52 умножить на 48?', additional_kwargs={}, response_metadata={}, id='ec1f3234-848c-4d2e-a8d2-2610903a064f'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '178236236', 'function': {'arguments': '{"expression":"52 * 48"}', 'name': 'calculator'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 143, 'total_tokens': 182, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'openai/gpt-oss-20b', 'id': 'chatcmpl-96i052mjw6crn1l2wctljs', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019be1ea-187e-72a2-b4bc-26185195b13c-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '52 * 48'}, 'id': '178236236', 'type': 'tool_call'}], usage_metadata={'input_tokens': 143, 'output_tokens': 39, 'total_tokens': 182, 'input_token_d

In [8]:
# Тест 2: Сложное выражение
query2 = "Вычисли (25 + 17) * 3 - 10"
print(f"Запрос: {query2}\n")

messages = agent_executor.invoke({'messages': [query2]})['messages']
pprint(messages)
print(f"\nОтвет агента: {messages[-1].content}")

Запрос: Вычисли (25 + 17) * 3 - 10

[HumanMessage(content='Вычисли (25 + 17) * 3 - 10', additional_kwargs={}, response_metadata={}, id='501b1881-556d-4047-9d0e-bddff29c7ca2'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '257950169', 'function': {'arguments': '{"expression":"(25 + 17) * 3 - 10"}', 'name': 'calculator'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 147, 'total_tokens': 200, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'openai/gpt-oss-20b', 'id': 'chatcmpl-wsa9utg8liiacm2pdd4y', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019be1ea-bed7-70b1-95f7-305a5e98ce0c-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '(25 + 17) * 3 - 10'}, 'id': '257950169', 'type': 'tool_call'}], usage_metadata={'input_tokens': 147, 'output_tokens': 53, 'total_tokens': 200, 'input

In [9]:
# Тест 3: Проверка работы калькулятора напрямую
from tools.tools import calculator

print("Тест функции калькулятора напрямую:")
test_expr = "2 ** 10"
result = calculator.invoke(test_expr)
print(f"{test_expr} = {result}")

Тест функции калькулятора напрямую:
2 ** 10 = 1024
